In [1]:
import torch
import torchaudio
import math

In [2]:
# CONSTANTS DEFINED

MU = 255 #8 Bit


In [3]:
# FUNCTIONS DEFINED

# MU LAW COMPANSION FUNCTION

def mu_law_encode(x:torch.Tensor,mu:int=MU):
  x = torch.clamp(x,-1.0,1.0) # clipping signals that falls out of range
  y = torch.sign(x) * torch.log1p(mu*torch.abs(x))/ math.log1p(mu) # output of this is going to be in the range of [-1,1]
  y = (y+1)/ 2 * mu # to make sure that the number of values to output is in the range of [0,255]
  return y.long()

def mu_law_decode(y:torch.Tensor,mu:int=MU):
  y = y.float()
  y = 2*(y/mu) -1
  x = torch.sign(y) * (1.0/mu) * ((1+mu)**torch.abs(y)-1)
  return torch.clamp(x,-1.0,1.0)

In [9]:
# GETTING THE DATASET

from torch.utils.data import Dataset, DataLoader
import os,random

class LJSpeechWaveform(Dataset):
  def __init__ (self,root,segment_len=16384,target_len=1024,sr_out=16000):
    self.ds = torchaudio.datasets.LJSPEECH(root=root,download=True)
    self.resample = torchaudio.transforms.Resample(orig_freq=22050,new_freq=sr_out)
    self.segment_len = segment_len
    self.target_len = target_len
    self.total_len = segment_len + target_len
    self.sr = sr_out

  def __len__(self):
    return len(self.ds)

  def __getitem__(self,idx):
    wav,sr,_,_ = self.ds[idx]# wav(1,N) i.e --> [[x1,x2,x3....]]
    if sr!=self.sr:
      wav = self.resample(wav)
    wav = wav.squeeze(0) # i.e -->[x1,x2,x3...]
    if wav.numel() < self.total_len+1:
      # pad if too short
      pad = self.total_len + 1 -wav.numel()
      wav = torch.nn.functional.pad(wav,(0,pad))
    start = random.randint(0,wav.numel()-(self.total_len+1))
    chunk = wav[start:start+self.total_len+1] # +1 for teacher-forcing shift and causal convolution

    codes = mu_law_encode(chunk) # (L,)
    x = codes [:-1] # input
    y = codes [1:]  # next step target

    return x,y #both are in int64 format and in range of [0..255]


In [5]:
# CAUSAL AND DILATED CONVOLUTION WITH GATED RESIDUAL BLOCKS

import torch.nn as nn
import torch.nn.functional as F

class CausalConv1d(nn.Conv1d):
  def __init__(self,in_ch,out_ch,kernel_size,dilation=1):
    padding = dilation * (kernel_size-1)
    super().__init__(in_ch,out_ch,kernel_size,padding = padding,dilation=dilation) # here whatever the padding is, it's applied both on the left and the right
    self.pad = padding # this creates the padding that makes sure that each element is convoluted over the same dilation and if the element is towards the end there will be padded zeros to replicate x_last--convolution-->x_last

  def forward(self,x):
    # x: Batch(B), Channels (C), Time (T) basically the last number whatever it is in the dimension of x should be in the middle
    y = super().forward(x)
    return y[...,:-self.pad] if self.pad!=0 else y # here since the length of the og X is len(X) and the padded one is len(X)+4 and convolution on the padded_X gives as len(conv(padded_x)) = padded_X- dilation we need to remove the extra padding convolution outputs

class ResidualBlock(nn.Module):
  def __init__(self,res_ch,skip_ch,kernel_size=2,dilation=1,cond_ch=None):
    super().__init__()
    self.dilated = CausalConv1d(res_ch,2*res_ch,kernel_size,dilation) # one half of h would go to the filter and the other to the gate, we need independent parameters for each hence we would double the channels so that one half can act as the filter values and the other half would act as the gate values
    self.res_out = nn.Conv1d(res_ch,res_ch,kernel_size=1) #residual 1x1
    self.skip_out = nn.Conv1d(res_ch,skip_ch,kernel_size=1) # skip 1x1

    # optional local conditioning (e.g, mel) (F0); kept None for now
    self.has_cond = cond_ch is not None
    if self.has_cond:
      self.cond = nn.Conv1d(cond_ch,2*res_ch,kernel_size=1)

  def forward(self,x,cond=None):
    h = self.dilated(x)
    if self.has_cond and cond is not None:
      # cond should be time aligned (B,cond_ch,T)
      h = h + self.cond(cond)
    a,b = h.chunk(2,dim=1) # create two of the same (B,res_ch,T)(B,res_ch,T)
    z = torch.tanh(a) * torch.sigmoid(b)
    skip = self.skip_out(z)
    res = self.res_out(z) + x # residual added to x for the next layer
    return res,skip




In [6]:
class InputEmbed(nn.Module):
  def __init__(self,n_classes=256,res_ch=64):
    super().__init__()
    self.embed = nn.Embedding(n_classes,res_ch)
    self.proj = nn.Conv1d(res_ch,res_ch,kernel_size=1)
  def forward(self, x_codes): # (B,T)
    e = self.embed(x_codes) # B, T, res_ch
    e = e.transpose(1,2)    # B, res_ch, T
    return self.proj(e)     # B, res_ch, T

In [7]:
class WaveNet(nn.Module):
  def __init__(self,n_classes=256,res_ch=64,skip_ch=256,n_layers=8,n_cycles=2,kernel_size=2):
    super().__init__()
    self.n_classes =n_classes
    self.input = InputEmbed(n_classes,res_ch)
    self.causal_in =CausalConv1d(res_ch,res_ch,kernel_size=kernel_size,dilation=1)

    dilations=[]
    for _ in range(n_cycles):
      for i in range(n_layers):
        dilations.append(2**i)
    self.blocks = nn.ModuleList([
        ResidualBlock(res_ch,skip_ch,kernel_size,d) for d in dilations
    ])

    self.post = nn.Sequential(
        nn.ReLU(),
        nn.Conv1d(skip_ch,skip_ch,kernel_size=1),
        nn.ReLU(),
        nn.Conv1d(skip_ch,n_classes,kernel_size=1)
    )

  @property
  def receptive_field(self):
    # RF = 1 +sum(dilation * (k-1)) across all distilled layers + initial causal k -1
    k=2
    rf = (k-1)
    for m in self.blocks:
      rf+=m.dilated.dilation[0]*(k-1)
    return rf +1

  def forward(self,x_codes):
    # emb  = F.one_hot(x_codes,num_classes=self.n_classes).float() # B, T, 256
    # emb  = emb.transpose(1,2) # B, 256, T

    # proj = F.linear(emb.transpose(1,2),weight=torch.randn(1,self.n_classes,device=emb.device)) # B T 1
    # x    = proj.transpose(1,2) # B 1 T

    h    =self.input(x_codes)
    h    = self.causal_in(h)

    skip_sum=0
    for block in self.blocks:
      h,s = block(h)
      skip_sum = s if isinstance(skip_sum,int) else (skip_sum + s)

    out = self.post(skip_sum) # B 256 T
    return out



In [12]:
from tqdm import tqdm

def train_one_epoch(model,loader,opt,device="cuda"):
  model.train()
  total =0

  for x,y in tqdm(loader,ncols=80):
    x = x.to(device) # (B,T)
    y = y.to(device) # (B,T)

    logits = model(x)
    loss = F.cross_entropy(logits,y)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    total+=loss.item()*x.size(0)
  return total/len(loader.dataset)

#wiring
def make_loader(root, bs=8, seg_len=16384, tgt_len=1024, sr=16000, workers=4):
    ds = LJSpeechWaveform(root, segment_len=seg_len, target_len=tgt_len, sr_out=sr)
    return DataLoader(ds, batch_size=bs, shuffle=True, num_workers=workers, pin_memory=True, drop_last=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = WaveNet(n_classes=256, res_ch=64, skip_ch=256, n_layers=8, n_cycles=2).to(device)
print("Receptive field (samples):", model.receptive_field)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
loader = make_loader("/content/data", bs=4)

for epoch in range(50):
    avg = train_one_epoch(model, loader, opt, device=device)
    print(f"epoch {epoch}: loss {avg:.3f}")

Receptive field (samples): 512


  0%|                                                  | 0/3275 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs

epoch 0: loss 3.181


  0%|                                                  | 0/3275 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see

epoch 1: loss 2.874


...

epoch 3: loss 2.759


  0%|                                                  | 0/3275 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs

epoch 4: loss 2.722


  0%|                                                  | 0/3275 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see

KeyboardInterrupt: 

In [11]:
@torch.no_grad()
def generate(model, seed_codes, n_samples=16000, device="cuda"):
    model.eval()
    x = seed_codes.to(device)  # (T,)
    x = x.unsqueeze(0)         # (1,T)
    out = [c.item() for c in x[0]]

    for _ in tqdm(range(n_samples), ncols=80):
        logits = model(x)       # (1,256,T)
        next_logits = logits[:, :, -1]         # (1,256)
        next_code = torch.argmax(next_logits, dim=1)  # greedy
        out.append(int(next_code.item()))
        x = torch.tensor(out[-model.receptive_field:], device=device).unsqueeze(0)  # keep last RF tokens

    return torch.tensor(out, dtype=torch.long)

# Usage:
seed = torch.full((1024,), 128)  # flat mid-level seed, or take real snippet from dataset
gen_codes = generate(model, seed, n_samples=16000)
wav = mu_law_decode(gen_codes).float().unsqueeze(0)  # (1, L)
torchaudio.save("gen.wav", wav, 16000)


100%|████████████████████████████████████| 16000/16000 [01:18<00:00, 202.65it/s]
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio

In [ ]:
sr = 16000
seconds = 30
total_needed = seconds * sr  # 480_000

# make a seed at least as long as the receptive field
seed = torch.full((max(1024, model.receptive_field),), 128)  # or use a real snippet

# we want total (seed + new) == 30s
n_new = max(0, total_needed - seed.numel())

gen_codes = generate(model, seed, n_samples=n_new, device=device)  # includes seed
wav = mu_law_decode(gen_codes).float().unsqueeze(0)  # (1, L)
torchaudio.save("gen_30s_with_seed.wav", wav, sr)
